# 01 - Lancer le traitement compliance NLP

Ce notebook execute le traitement local sur les PDF du dossier `data/raw` et genere les resultats dans `outputs/analysis/results.json`.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
elif not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from compliance_nlp import (
    analyze_directory,
    load_generic_detection_rules,
    load_section_definitions,
    load_whitelist_terms,
    results_to_dataframe,
    summarize_results,
)

import pandas as pd

## Parametres d'execution

In [ ]:
input_dir = repo_root / "data" / "raw"
results_path = repo_root / "outputs" / "analysis" / "results.json"
findings_csv_path = repo_root / "outputs" / "analysis" / "findings.csv"

sections_path = repo_root / "configs" / "sections.csv"
generic_rules_path = repo_root / "configs" / "generic_detection_rules.csv"
whitelist_path = repo_root / "configs" / "article9_whitelist.csv"

ENABLE_GENERIC_BRANCH = True
ENABLE_SPACY_BRANCH = True
ENABLE_GLINER_BRANCH = False
MODEL_CACHE_DIR = r"D:\Workspaces\ModelCache"
MODEL_STORE_DIR = r"D:\Workspaces\modelStore"
SPACY_MODEL = rf"{MODEL_STORE_DIR}\fr_core_news_sm"
GLINER_MODEL = rf"{MODEL_STORE_DIR}\gliner_multi-v2.1"
GLINER_SOURCE_MODEL = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.50
GLINER_LOCAL_FILES_ONLY = True
GLINER_LABELS = [
    "donnee de sante",
    "maladie",
    "pathologie",
    "etat de sante",
    "condition medicale",
    "probleme de sante",
    "opinion politique",
    "conviction religieuse",
    "appartenance syndicale",
    "orientation sexuelle",
    "origine ethnique",
    "donnee genetique",
    "donnee biometrique",
    "clause beneficiaire imprecise",
    "conseil non professionnel",
    "promesse de performance",
]

enabled_branches = tuple(
    branch
    for branch, enabled in {
        "generic": ENABLE_GENERIC_BRANCH,
        "spacy": ENABLE_SPACY_BRANCH,
        "gliner": ENABLE_GLINER_BRANCH,
    }.items()
    if enabled
)

params = {
    "input_dir": str(input_dir),
    "results_path": str(results_path),
    "findings_csv_path": str(findings_csv_path),
    "sections_path": str(sections_path),
    "generic_rules_path": str(generic_rules_path),
    "whitelist_path": str(whitelist_path),
    "enabled_branches": enabled_branches,
    "model_cache_dir": MODEL_CACHE_DIR,
    "model_store_dir": MODEL_STORE_DIR,
    "spacy_model": SPACY_MODEL,
    "gliner_model": GLINER_MODEL,
    "gliner_source_model": GLINER_SOURCE_MODEL,
    "gliner_threshold": GLINER_THRESHOLD,
    "gliner_local_files_only": GLINER_LOCAL_FILES_ONLY,
    "gliner_labels": GLINER_LABELS,
}
params

## Verification des fichiers d'entree

In [ ]:
pdf_files = sorted(input_dir.glob("*.pdf"))
pd.DataFrame({"pdf": [pdf.name for pdf in pdf_files], "size_bytes": [pdf.stat().st_size for pdf in pdf_files]})

## Verification des referentiels de controle

In [ ]:
sections = load_section_definitions(sections_path)
generic_rules = load_generic_detection_rules(generic_rules_path)
whitelist_terms = load_whitelist_terms(whitelist_path)

pd.DataFrame([
    {"referentiel": "sections", "count": len(sections)},
    {"referentiel": "central_detection_rules", "count": len(generic_rules)},
    {"referentiel": "central_article9_rules", "count": sum(rule.rule_scope == "article9" for rule in generic_rules)},
    {"referentiel": "central_general_rules", "count": sum(rule.rule_scope == "general" for rule in generic_rules)},
    {"referentiel": "article9_whitelist", "count": len(whitelist_terms)},
])

In [ ]:
pd.DataFrame([
    {
        "rule_id": rule.rule_id,
        "rule_scope": rule.rule_scope,
        "regulatory_family": rule.regulatory_family,
        "section_scope": " | ".join(rule.section_scope),
        "category": rule.category,
        "label": rule.label,
        "terms": " | ".join(rule.terms),
        "synonyms": " | ".join(rule.synonyms),
        "base_score": rule.base_score,
        "fuzzy_threshold": rule.fuzzy_threshold,
        "applies_whitelist": rule.applies_whitelist,
    }
    for rule in generic_rules
])

## Execution du traitement

In [ ]:
results = analyze_directory(
    input_dir,
    output_path=results_path,
    sections_path=sections_path,
    generic_rules_path=generic_rules_path,
    whitelist_path=whitelist_path,
    enabled_branches=enabled_branches,
    spacy_model=SPACY_MODEL,
    gliner_model=GLINER_MODEL,
    gliner_cache_dir=MODEL_CACHE_DIR,
    gliner_source_model=GLINER_SOURCE_MODEL,
    gliner_labels=GLINER_LABELS,
    gliner_threshold=GLINER_THRESHOLD,
    gliner_local_files_only=GLINER_LOCAL_FILES_ONLY,
)

summary = summarize_results(results)
summary

## Resultats generes

In [ ]:
df = results_to_dataframe(results)
findings_csv_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(findings_csv_path, index=False, encoding="utf-8-sig")

{
    "json_results": str(results_path),
    "csv_findings": str(findings_csv_path),
    "rows": len(df),
}

In [ ]:
pd.DataFrame([
    {
        "document_name": result.document_name,
        "enabled_branches": " | ".join(result.metadata.get("enabled_branches", [])),
        "generic_finding_count": result.metadata.get("generic_finding_count"),
        "spacy_finding_count": result.metadata.get("spacy_finding_count"),
        "gliner_finding_count": result.metadata.get("gliner_finding_count"),
        "generic_max_score": result.metadata.get("generic_max_score"),
        "spacy_max_score": result.metadata.get("spacy_max_score"),
        "gliner_max_score": result.metadata.get("gliner_max_score"),
        "branch_errors": result.metadata.get("branch_errors"),
    }
    for result in results
])

In [ ]:
df[[
    "document_name",
    "detection_engine",
    "code",
    "section",
    "category",
    "matched_term",
    "detection_type",
    "score",
    "generic_score",
    "spacy_score",
    "gliner_score",
    "severity",
    "alert_level",
]].sort_values(["document_name", "score"], ascending=[True, False], na_position="last")